# 🕷️ Spidey-Tracker OS: NYPD & Daily Bugle Threat Verification Engine

```text
       /\             /\
      /  \  __   __  /  \
     /    \(  \_/  )/    \
    /  /\  \       /  /\  \
   /  /  \  \     /  /  \  \
  (  (    \  )   (  /    )  )
   \  \    \(     )/    /  /
    \  \___/  ===  \___/  /
     \_      /   \      _/
       \____/     \____/
```

**Terminal Clearance:** Level 4 (Stark Industries & NYPD Web-Net Access)  
**Dataset Ground-Truth:** [Kaggle — Spidey Tracker (86,631 Sightings)](https://www.kaggle.com/datasets/umuttuygurr/spidey-tracker-spiderman-dataset)  
**Live Web HUD:** [https://shiventhakker.github.io/ML-Assignment/](https://shiventhakker.github.io/ML-Assignment/)

### 🎯 Mission Briefing:
1. **Calibrate Spider-Sense (K-NN):** Filter authentic Spider-Man emergencies from Mysterio illusions, cosplayers, and Daily Bugle hoaxes. Track progressive accuracy gains through 7 suit-upgrade cycles.
2. **Patrol Grid Analysis (K-Means):** Unsupervised clustering of sighting hot zones across New York's 5 boroughs.
3. **Mission 05 Detective Solve:** Triangulate Peter Parker's secret operating sanctuary using daily departure/return telemetry.

## 1. System Initialization & Tactical Imports

In [ ]:
# Install kagglehub to connect to the Spidey-Tracker data feed
!pip install -q kagglehub

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Marvel Spider-Man Themed Color Palette
SPIDEY_RED = '#D82C2C'       # Real Spider-Man / Verified Features
WEB_BLUE = '#094074'         # Web-Shooter Blue / Baseline Curves
SYMB_BLACK = '#1B1B1B'       # Symbiote Dark / Text / Cluster Centers
SPIDER_GOLD = '#F4D03F'      # Spider-Sense Gold / Optimal Points / Base
MYSTERIO_PURPLE = '#8E44AD'  # Daily Bugle Hoaxes / Fake Sightings
GOBLIN_GREEN = '#27AE60'     # Validated Safe / Verified Sightings

# Configure tactical plotting aesthetics
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = SYMB_BLACK
plt.rcParams['axes.linewidth'] = 1.2
sns.set_style('whitegrid', {'grid.color': '#e0e0e0', 'grid.linestyle': '--'})

# Helper to format column strings to clean Title Case
def format_title(text):
    return str(text).replace('_', ' ').replace('-', ' ').strip().title()

print("[SPIDER-SENSE TELEMETRY]")
print("📡 All tactical libraries linked. Ready to stream NYC scanner logs.")

## 2. Ingesting Daily Bugle & NYPD Scanner Logs

In [ ]:
import kagglehub

print("[SPIDER-SENSE TELEMETRY]")
print("📡 Connecting to Kaggle Web-Net satellite feed...")
path = kagglehub.dataset_download("umuttuygurr/spidey-tracker-spiderman-dataset")
print(f"✅ Data payload received at: {path}")

# Scan directory for all tactical intelligence tables
print("\n📂 Intelligence Manifest:")
for root, dirs, files in os.walk(path):
    for file in files:
        filepath = os.path.join(root, file)
        print(f"  🕸️ {file:26s} ({os.path.getsize(filepath) / 1024:.0f} KB)")

In [ ]:
# Find and load the main sightings database
csv_path = os.path.join(path, 'sightings.csv')
if not os.path.exists(csv_path):
    for root, dirs, files in os.walk(path):
        if 'sightings.csv' in files:
            csv_path = os.path.join(root, 'sightings.csv')
            break

print(f"[SPIDER-SENSE TELEMETRY] Loading master sightings register from: {csv_path}")
df = pd.read_csv(csv_path)
print(f"✅ Master Dossier Loaded: {df.shape[0]:,} sighting reports across {df.shape[1]} attributes.")
df.head(3)

## 3. Exploratory Data Intelligence (Scanning the 5 Boroughs)

In [ ]:
# Check missing values and telemetry integrity
missing_counts = df.isnull().sum()
total_missing = missing_counts.sum()

print("[SPIDER-SENSE TELEMETRY] Telemetry Integrity Audit:")
print(f"  Total Data Fields: {df.size:,}")
print(f"  Missing Readings:  {total_missing:,}")
if total_missing > 0:
    print("\nColumns requiring sensor interpolation:")
    print(missing_counts[missing_counts > 0])

In [ ]:
# Target Distribution: Genuine Web-Slinger vs Hoax Reports
verified_count = df['is_verified'].sum()
hoax_count = len(df) - verified_count
verified_ratio = df['is_verified'].mean()

print("[SPIDER-SENSE TELEMETRY] Target Distribution Analysis:")
print(f"  🕷️ Confirmed Spider-Man Sightings: {verified_count:,} ({verified_ratio:.2%})")
print(f"  🗞️ Daily Bugle Hoaxes / Fake News:  {hoax_count:,} ({1 - verified_ratio:.2%})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Bar chart with Spidey Palette
axes[0].bar(['Daily Bugle Hoax', 'Confirmed Spidey'], [hoax_count, verified_count],
            color=[MYSTERIO_PURPLE, SPIDEY_RED], edgecolor=SYMB_BLACK, linewidth=1.5, width=0.55)
axes[0].set_title('Verified Hero Vs Daily Bugle Hoax Reports', fontsize=12, fontweight='bold', pad=10)
axes[0].set_ylabel('Number Of Reports', fontsize=11)
for i, v in enumerate([hoax_count, verified_count]):
    axes[0].text(i, v + 1000, f"{v:,}\n({v/len(df):.1%})", ha='center', fontweight='bold', fontsize=10)

# Donut chart
wedges, texts, autotexts = axes[1].pie(
    [hoax_count, verified_count],
    labels=['Daily Bugle Hoaxes', 'Confirmed Spider-Man'],
    colors=[MYSTERIO_PURPLE, SPIDEY_RED],
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(width=0.45, edgecolor=SYMB_BLACK, linewidth=1.5)
)
for at in autotexts:
    at.set_color('white')
    at.set_fontweight('bold')
axes[1].set_title('Sighting Authenticity Proportion', fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()

In [ ]:
# Feature Correlation Heatmap (Spider-Sense Feature Interlink)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

clean_cols_map = {col: format_title(col) for col in numeric_cols}
corr = df[numeric_cols].rename(columns=clean_cols_map, index=clean_cols_map).corr()

plt.figure(figsize=(15, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(240, 10, as_cmap=True)

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap, center=0,
            square=True, linewidths=0.75, linecolor='white',
            cbar_kws={'label': 'Pearson Correlation Coefficient'})
plt.title('Spider-Sense Telemetry — Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of Key Sensor Readings
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
axes = axes.flatten()

plot_cols = [c for c in numeric_cols if 'id' not in c.lower()][:6]
for i, col in enumerate(plot_cols):
    clean_name = format_title(col)
    axes[i].hist(df[col].dropna(), bins=45, color=WEB_BLUE, edgecolor=SYMB_BLACK, alpha=0.85, linewidth=0.8)
    axes[i].set_title(clean_name, fontsize=11, fontweight='bold')
    axes[i].set_xlabel(clean_name, fontsize=10)
    axes[i].set_ylabel('Report Frequency', fontsize=10)

plt.suptitle('Sensor Telemetry Distributions Across NYC Sightings', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Preprocessing: Filtering Mysterio's Static

Before training our Spider-Sense classifier, we prepare the telemetry:
1. Strip tracking identifiers (`id`, `target`, `split`) to prevent data leakage.
2. Impute sensor dropouts with feature medians.
3. Preserve the official stratified partition (`train` vs `test`).

In [ ]:
# Define prediction target: is_verified (1 = Real Spidey, 0 = Hoax)
target = 'is_verified'
y = df[target]

# Retain predictive numerical features
drop_keywords = ['id', 'target', 'verified', 'verification', 'split']
feature_cols = [c for c in numeric_cols
                if not any(kw in c.lower() for kw in drop_keywords)]

print(f"[SPIDER-SENSE TELEMETRY] Selected {len(feature_cols)} tactical features:")
for col in feature_cols:
    print(f"  🕷️ {format_title(col)}")

X = df[feature_cols].copy()

# Handle missing readings via median imputation
missing_before = X.isnull().sum().sum()
X = X.fillna(X.median())
print(f"\n✅ Sensor dropouts resolved: {missing_before:,} missing values imputed.")

In [ ]:
# Train/Test Split
if 'split' in df.columns:
    train_mask = df['split'] == 'train'
    test_mask = df['split'] == 'test'
    X_train, X_test = X[train_mask].values, X[test_mask].values
    y_train, y_test = y[train_mask].values, y[test_mask].values
    print("[SPIDER-SENSE TELEMETRY] Using official stratified partition:")
else:
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X.values, y.values, test_size=0.2, random_state=42, stratify=y)
    print("[SPIDER-SENSE TELEMETRY] Using 80/20 stratified split:")

print(f"  🏋️ Training Archive:   {X_train.shape[0]:,} reports ({y_train.mean():.2%} verified)")
print(f"  🎯 Field Test Chamber:  {X_test.shape[0]:,} reports ({y_test.mean():.2%} verified)")

## 5. Calibrating Spider-Sense (K-NN Classification)

We progressively upgrade the Spider-Sense algorithm across **7 distinct suit-upgrade stages**, measuring and verifying every single improvement.

---

### 🧵 Suit Upgrade #1: Homemade Hoodie Suit (Baseline Model)
The raw uncalibrated prototype: default scikit-learn parameters ($K=5$, uniform voting, Minkowski distance) with **no feature scaling**.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

results = []

# Baseline Model
knn_baseline = KNeighborsClassifier(n_neighbors=5)
knn_baseline.fit(X_train, y_train)
y_pred_baseline = knn_baseline.predict(X_test)

acc_baseline = accuracy_score(y_test, y_pred_baseline)
results.append({
    'Step': '1. Homemade Hoodie Suit (Baseline)',
    'Accuracy': acc_baseline,
    'Suit': 'Homemade Hoodie',
    'Details': 'K = 5, Uniform, Minkowski, No Scaling'
})

print("[SPIDER-SENSE TELEMETRY]")
print(f"🧵 SUIT UPGRADE #1: Homemade Hoodie Suit")
print(f"📊 Baseline Accuracy: {acc_baseline:.4f}")
print("⚠️ Distortion Warning: Unscaled feature magnitudes dominate distance metrics!")
print(f"\n{classification_report(y_test, y_pred_baseline, target_names=['Daily Bugle Hoax', 'Real Spidey'])}")

In [ ]:
# Confusion Matrix with Quad Subtitles
fig, ax = plt.subplots(figsize=(7, 5.5))
cm_base = confusion_matrix(y_test, y_pred_baseline)

sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Daily Bugle Hoax', 'Real Spidey'],
            yticklabels=['Daily Bugle Hoax', 'Real Spidey'],
            annot_kws={'size': 12, 'weight': 'bold'},
            cbar=False)

ax.set_title(f'Homemade Hoodie Suit — Accuracy: {acc_baseline:.4f}\n'
             f'Top-Left: Hoax Disproved 🗞️ | Top-Right: Imposter Mistaken 🎭\n'
             f'Bottom-Left: Spidey Slipped Away 💨 | Bottom-Right: Spidey Confirmed 🕷️',
             fontsize=10, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Sighting Status', fontsize=11, fontweight='bold')
ax.set_ylabel('Actual Sighting Status', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

### 🕸️ Suit Upgrade #2: Classic Spidey Suit (StandardScaler Activated)

K-NN computes spatial distances between sightings. Because features like `distance_to_nearest_crime_m` (in thousands of meters) dwarf `tracker_confidence` (0 to 1), we apply `StandardScaler` ($\mu=0, \sigma=1$) to equalize every sensor dimension.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled)

acc_scaled = accuracy_score(y_test, y_pred_scaled)
results.append({
    'Step': '2. Classic Spidey Suit (Scaling)',
    'Accuracy': acc_scaled,
    'Suit': 'Classic Spidey',
    'Details': 'StandardScaler Normalization'
})

change = acc_scaled - acc_baseline
print("[SPIDER-SENSE TELEMETRY]")
print(f"🕸️ SUIT UPGRADE #2: Classic Spidey Suit (Feature Scaling)")
print(f"⚡ Feature distortion neutralized across all {len(feature_cols)} dimensions.")
print(f"📈 Spider-Sense Accuracy: {acc_scaled:.4f} ({'+' if change >= 0 else ''}{change:.4f} gain)")

### 🎯 Suit Upgrade #3: Advanced Web-Shooters (Optimal Neighborhood Radius K)

The default $K=5$ is an arbitrary choice. We sweep $K=1$ through $K=30$ to find the neighborhood boundary that balances bias and variance.

In [ ]:
# Sweep K from 1 to 30
k_range = range(1, 31)
k_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    k_scores.append(knn.score(X_test_scaled, y_test))

best_k = list(k_range)[np.argmax(k_scores)]
best_k_score = max(k_scores)

# Plot K vs Accuracy with Spider-Sense theme
plt.figure(figsize=(11, 5))
plt.plot(k_range, k_scores, 'o-', color=WEB_BLUE, linewidth=2, markersize=5, label='K-NN Neighborhood Accuracy')
plt.axvline(x=best_k, color=SPIDEY_RED, linestyle='--', linewidth=2,
            label=f'Optimal K = {best_k} (Accuracy: {best_k_score:.4f})')
plt.scatter([best_k], [best_k_score], color=SPIDER_GOLD, s=180, edgecolors=SYMB_BLACK, linewidth=1.5, zorder=5)

plt.xlabel('Number Of Neighbors (K)', fontsize=11, fontweight='bold')
plt.ylabel('Model Accuracy', fontsize=11, fontweight='bold')
plt.title('Calibrating Spider-Sense Radius — Accuracy Vs Neighborhood Size (K)', fontsize=12, fontweight='bold', pad=10)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

results.append({
    'Step': f'3. Advanced Web-Shooters (K={best_k})',
    'Accuracy': best_k_score,
    'Suit': 'Advanced Web-Shooters',
    'Details': f'Optimal Radius K = {best_k}'
})

change = best_k_score - acc_scaled
print("[SPIDER-SENSE TELEMETRY]")
print(f"🎯 SUIT UPGRADE #3: Optimal Neighborhood Radius Locked at K = {best_k}")
print(f"📈 Spider-Sense Accuracy: {best_k_score:.4f} ({'+' if change >= 0 else ''}{change:.4f} gain)")

### 🏙️ Suit Upgrade #4: Spider-Armor MK I (Manhattan City-Grid Metric)

New York City is built on a rectangular street grid. Straight-line (Euclidean) distance doesn't capture urban navigation as naturally as **Manhattan distance (L1 norm)**. We compare Euclidean, Manhattan, and Chebyshev metrics.

In [ ]:
metrics = ['euclidean', 'manhattan', 'chebyshev']
metric_scores = {}

for m in metrics:
    knn = KNeighborsClassifier(n_neighbors=best_k, metric=m)
    knn.fit(X_train_scaled, y_train)
    score = knn.score(X_test_scaled, y_test)
    metric_scores[m] = score
    print(f"  🕸️ Metric: {m.title():12s} -> Accuracy: {score:.4f}")

best_metric = max(metric_scores, key=metric_scores.get)
best_metric_score = metric_scores[best_metric]

results.append({
    'Step': f'4. Spider-Armor MK I ({best_metric.title()})',
    'Accuracy': best_metric_score,
    'Suit': 'Spider-Armor MK I',
    'Details': f'Distance Formula: {best_metric.title()}'
})

# Compare metrics visually
plt.figure(figsize=(9, 4.5))
metric_names = [m.title() for m in metrics]
bar_colors = [SPIDEY_RED if m == best_metric else WEB_BLUE for m in metrics]

bars = plt.bar(metric_names, [metric_scores[m] for m in metrics],
               color=bar_colors, edgecolor=SYMB_BLACK, linewidth=1.2, width=0.5)
plt.xlabel('Distance Metric Formulation', fontsize=11, fontweight='bold')
plt.ylabel('Model Accuracy', fontsize=11, fontweight='bold')
plt.title('Urban Topology Comparison — Distance Metric Impact', fontsize=12, fontweight='bold', pad=10)

for i, m in enumerate(metrics):
    plt.text(i, metric_scores[m] + 0.001, f"{metric_scores[m]:.4f}", ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

change = best_metric_score - best_k_score
print("[SPIDER-SENSE TELEMETRY]")
print(f"🏙️ SUIT UPGRADE #4: Urban Grid Metric Selected: {best_metric.title()}")
print(f"📈 Spider-Sense Accuracy: {best_metric_score:.4f} ({'+' if change >= 0 else ''}{change:.4f} gain)")

### ⚖️ Suit Upgrade #5: Stealth Big Time Suit (Distance-Decay Weighting)

In standard K-NN, all $K$ neighbors cast an equal vote regardless of distance. By switching to `weights='distance'`, closer witness accounts have proportionally greater influence.

In [ ]:
weight_options = ['uniform', 'distance']
weight_scores = {}

for w in weight_options:
    knn = KNeighborsClassifier(n_neighbors=best_k, metric=best_metric, weights=w)
    knn.fit(X_train_scaled, y_train)
    score = knn.score(X_test_scaled, y_test)
    weight_scores[w] = score
    print(f"  🕸️ Weighting: {w.title():10s} -> Accuracy: {score:.4f}")

best_weight = max(weight_scores, key=weight_scores.get)
best_weight_score = weight_scores[best_weight]

results.append({
    'Step': f'5. Stealth Big Time ({best_weight.title()})',
    'Accuracy': best_weight_score,
    'Suit': 'Stealth Big Time',
    'Details': f'Voting Weighting: {best_weight.title()}'
})

change = best_weight_score - best_metric_score
print("[SPIDER-SENSE TELEMETRY]")
print(f"⚖️ SUIT UPGRADE #5: Proximity Weighting Locked: {best_weight.title()}")
print(f"📈 Spider-Sense Accuracy: {best_weight_score:.4f} ({'+' if change >= 0 else ''}{change:.4f} gain)")

### 🔬 Suit Upgrade #6: Superior Spider Suit (Karen AI Feature Selection)

Noisy sensor signals (e.g. ambient crowd density or redundant timestamps) confuse distance calculations. We run ANOVA F-tests via `SelectKBest` to keep only the highest-signal attributes.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

k_features_range = range(2, len(feature_cols) + 1)
feature_scores = []

for k_feat in k_features_range:
    selector = SelectKBest(f_classif, k=k_feat)
    X_train_sel = selector.fit_transform(X_train_scaled, y_train)
    X_test_sel = selector.transform(X_test_scaled)

    knn = KNeighborsClassifier(n_neighbors=best_k, metric=best_metric, weights=best_weight)
    knn.fit(X_train_sel, y_train)
    feature_scores.append(knn.score(X_test_sel, y_test))

best_n_features = list(k_features_range)[np.argmax(feature_scores)]
best_feat_score = max(feature_scores)

# Plot feature curve
plt.figure(figsize=(11, 5))
plt.plot(list(k_features_range), feature_scores, 'o-', color=WEB_BLUE, linewidth=2, markersize=5)
plt.axvline(x=best_n_features, color=SPIDEY_RED, linestyle='--', linewidth=2,
            label=f'Optimal Feature Count = {best_n_features}')
plt.scatter([best_n_features], [best_feat_score], color=SPIDER_GOLD, s=180, edgecolors=SYMB_BLACK, linewidth=1.5, zorder=5)

plt.xlabel('Number Of Selected Features', fontsize=11, fontweight='bold')
plt.ylabel('Model Accuracy', fontsize=11, fontweight='bold')
plt.title('Sensor Pruning — Accuracy Vs Retained Feature Count', fontsize=12, fontweight='bold', pad=10)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Rank and display selected features
selector_final = SelectKBest(f_classif, k=best_n_features)
X_train_selected = selector_final.fit_transform(X_train_scaled, y_train)
X_test_selected = selector_final.transform(X_test_scaled)

selected_mask = selector_final.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

print(f"[SPIDER-SENSE TELEMETRY] Karen AI retained {best_n_features}/{len(feature_cols)} tactical features:")
for f in selected_features:
    print(f"  ✅ {format_title(f)}")

# Feature Importance Horizontal Bar Chart
f_scores = selector_final.scores_
feat_importance = pd.DataFrame({
    'Feature': [format_title(f) for f in feature_cols],
    'Original': feature_cols,
    'F-Score': f_scores
}).sort_values('F-Score', ascending=True)

plt.figure(figsize=(11, max(4.5, len(feature_cols) * 0.42)))
bar_colors = [SPIDEY_RED if f in selected_features else '#BDC3C7' for f in feat_importance['Original']]
plt.barh(feat_importance['Feature'], feat_importance['F-Score'], color=bar_colors, edgecolor=SYMB_BLACK, linewidth=0.8)

plt.xlabel('ANOVA F-Score (Signal Strength)', fontsize=11, fontweight='bold')
plt.ylabel('Telemetry Sensor Feature', fontsize=11, fontweight='bold')
plt.title('Feature Signal Rankings (Red = Retained By Karen AI, Gray = Pruned)', fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

results.append({
    'Step': f'6. Superior Spider Suit ({best_n_features} Features)',
    'Accuracy': best_feat_score,
    'Suit': 'Superior Spider',
    'Details': f'Retained Top {best_n_features} Features'
})

change = best_feat_score - best_weight_score
print("[SPIDER-SENSE TELEMETRY]")
print(f"🔬 SUIT UPGRADE #6: Sensory Pruning Complete ({best_n_features} Features)")
print(f"📈 Spider-Sense Accuracy: {best_feat_score:.4f} ({'+' if change >= 0 else ''}{change:.4f} gain)")

### 🛡️ Suit Upgrade #7: Iron Spider Stark-Tech Suit (5-Fold Cross-Validation)

To guarantee that our tuned algorithm doesn't suffer from overfitting to a lucky test split, we execute **5-Fold Stratified Cross-Validation** across the entire training corpus.

In [ ]:
from sklearn.model_selection import cross_val_score

knn_final = KNeighborsClassifier(n_neighbors=best_k, metric=best_metric, weights=best_weight)
selector_cv = SelectKBest(f_classif, k=best_n_features)
X_train_cv = selector_cv.fit_transform(X_train_scaled, y_train)

cv_scores = cross_val_score(knn_final, X_train_cv, y_train, cv=5, scoring='accuracy')

print("[SPIDER-SENSE TELEMETRY] 5-Fold Stratified Validation Results:")
for i, score in enumerate(cv_scores):
    print(f"  🕸️ Fold {i+1}: Accuracy = {score:.4f}")
print(f"\n  ⭐ Mean Cross-Validation Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

results.append({
    'Step': '7. Iron Spider Stark Suit (5-Fold CV)',
    'Accuracy': cv_scores.mean(),
    'Suit': 'Iron Spider Stark-Tech',
    'Details': f'5-Fold CV Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}'
})

In [ ]:
# Final Held-Out Evaluation & Confusion Matrix Showdown
X_test_final = selector_cv.transform(X_test_scaled)
knn_final.fit(X_train_cv, y_train)
y_pred_final = knn_final.predict(X_test_final)

acc_final = accuracy_score(y_test, y_pred_final)

print(f"[SPIDER-SENSE TELEMETRY] 🛡️ Final Iron Spider Suit Test Accuracy: {acc_final:.4f}")
print(f"\n{classification_report(y_test, y_pred_final, target_names=['Daily Bugle Hoax', 'Real Spidey'])}")

# Side-by-Side Confusion Matrix Showdown
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cm_base = confusion_matrix(y_test, y_pred_baseline)
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=['Daily Bugle Hoax', 'Real Spidey'],
            yticklabels=['Daily Bugle Hoax', 'Real Spidey'],
            annot_kws={'size': 12, 'weight': 'bold'}, cbar=False)
axes[0].set_title(f'Homemade Hoodie Suit (Baseline: {acc_baseline:.4f})\n'
                  f'Top-Left: Hoax Disproved 🗞️ | Top-Right: Imposter Mistaken 🎭\n'
                  f'Bottom-Left: Spidey Slipped Away 💨 | Bottom-Right: Spidey Confirmed 🕷️',
                  fontsize=10, fontweight='bold', pad=10)
axes[0].set_xlabel('Predicted Status', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Actual Status', fontsize=11, fontweight='bold')

cm_final = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Daily Bugle Hoax', 'Real Spidey'],
            yticklabels=['Daily Bugle Hoax', 'Real Spidey'],
            annot_kws={'size': 12, 'weight': 'bold'}, cbar=False)
axes[1].set_title(f'Iron Spider Stark-Tech Suit (Optimized: {acc_final:.4f})\n'
                  f'Top-Left: Hoax Disproved 🗞️ | Top-Right: Imposter Mistaken 🎭\n'
                  f'Bottom-Left: Spidey Slipped Away 💨 | Bottom-Right: Spidey Confirmed 🕷️',
                  fontsize=10, fontweight='bold', pad=10)
axes[1].set_xlabel('Predicted Status', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Actual Status', fontsize=11, fontweight='bold')

plt.suptitle('Tactical Evolution — Baseline Vs Stark-Tech Confusion Matrices', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Patrol Grid Analysis (K-Means Clustering) & Mission 05

We deploy **unsupervised K-Means clustering** to partition New York City into optimal tactical patrol sectors, then analyze daily departure/return patterns to **solve Mission 05: Unmasking Spider-Man's Secret Operating Base**.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Extract coordinates
coords = df[['latitude', 'longitude']].dropna()
scaler_km = StandardScaler()
coords_scaled = scaler_km.fit_transform(coords)

print(f"[SPIDER-SENSE TELEMETRY] Clustering {len(coords):,} geographic sightings across NYC.")

In [ ]:
# Elbow Method & Silhouette Analysis
K_clusters_range = range(2, 13)
inertias = []
silhouette_scores = []

for k in K_clusters_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(coords_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(coords_scaled, labels, sample_size=min(5000, len(coords)))
    silhouette_scores.append(sil)

best_k_means = list(K_clusters_range)[np.argmax(silhouette_scores)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Curve
axes[0].plot(K_clusters_range, inertias, 'o-', color=WEB_BLUE, linewidth=2, markersize=5)
axes[0].set_xlabel('Number Of Patrol Sectors (K)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Inertia (Within-Cluster Sum Of Squares)', fontsize=11, fontweight='bold')
axes[0].set_title('Optimal Patrol Grid — Elbow Method', fontsize=12, fontweight='bold', pad=10)

# Silhouette Curve
axes[1].plot(K_clusters_range, silhouette_scores, 'o-', color=SPIDEY_RED, linewidth=2, markersize=5)
axes[1].axvline(x=best_k_means, color=SPIDER_GOLD, linestyle='--', linewidth=2,
                label=f'Optimal K = {best_k_means}')
axes[1].scatter([best_k_means], [max(silhouette_scores)], color=SPIDER_GOLD, s=180, edgecolors=SYMB_BLACK, zorder=5)
axes[1].set_xlabel('Number Of Patrol Sectors (K)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Silhouette Score', fontsize=11, fontweight='bold')
axes[1].set_title('Sector Separation — Silhouette Score Vs K', fontsize=12, fontweight='bold', pad=10)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"[SPIDER-SENSE TELEMETRY] Optimal city patrol grid locked at K = {best_k_means} clusters.")

In [ ]:
# Execute final K-Means Clustering
km_final = KMeans(n_clusters=best_k_means, init='k-means++', random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(coords_scaled)
centers = scaler_km.inverse_transform(km_final.cluster_centers_)

# Plot patrol clusters
plt.figure(figsize=(12, 9))
scatter = plt.scatter(coords['longitude'], coords['latitude'],
                      c=cluster_labels, cmap='coolwarm', alpha=0.35, s=6)
plt.scatter(centers[:, 1], centers[:, 0],
            c=SPIDER_GOLD, marker='X', s=320, edgecolors=SYMB_BLACK, linewidth=2,
            label='Patrol Sector Centroids', zorder=10)

plt.xlabel('Geographic Longitude', fontsize=11, fontweight='bold')
plt.ylabel('Geographic Latitude', fontsize=11, fontweight='bold')
plt.title(f'K-Means Tactical Patrol Sectors Across New York (K = {best_k_means})', fontsize=13, fontweight='bold', pad=12)
plt.colorbar(scatter, label='Patrol Sector ID')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 🕵️ Mission 05: Unmasking Spider-Man's Secret Operating Base (Chronometric Behavioral Triangulation)

The Kaggle dataset challenged researchers with an unscored mission without an answer key: **Infer Spider-Man's hidden operating sanctuary from raw behavioral telemetry.**

#### The Behavioral Principle of Vigilante Commuting
While midday sightings reflect **reactive crime hotspots** (bank heists in Midtown Manhattan, high-speed car chases in Downtown), **Peter Parker is fundamentally a human citizen who must sleep in a bedroom**. 
Every human daily cycle obeys a strict closed-loop thermodynamic routine:
1. **The Morning Departure Origin ($t_{\text{first}}$):** The first confirmed sighting of each morning marks where he puts on the suit and exits his home window.
2. **The Nighttime Return Terminal ($t_{\text{last}}$):** The final confirmed sighting of each night marks where he concludes his patrol and heads home to rest.
3. **The Nocturnal Rest Window (01:00 AM – 05:00 AM):** During deep sleep hours, movement drops to zero, clustering in the immediate residential neighborhood.
4. **Operational Concealment Demographics (`locations.csv`):** A vigilante cannot operate from Times Square or an active NYPD precinct. The sanctuary requires **high residential housing density** (rowhouses/apartments) and **low police patrol density** (minimal radar surveillance).

We crack this mystery through a systematic 4-phase deductive pipeline combining chronometry, nocturnal clustering, demographic profiling, and spatial $L_1$ median triangulation.

#### Phase 1: Chronometric Directional Telemetry (Morning Departures vs. Nighttime Returns)

We group verified sightings by calendar date and isolate:
- **Daily Morning Patrol Departures:** The earliest verified sighting of each day: $t_0(D) = \min_{i \in D} (t_i)$
- **Daily Nighttime Returns:** The final verified sighting before Spider-Man disappears for the night: $t_{\text{end}}(D) = \max_{i \in D} (t_i)$

In [ ]:
# Phase 1: Chronometric Directional Analysis (First & Last Appearances by District)
verified_df = df[df['is_verified'] == 1].copy()
verified_df['parsed_time'] = pd.to_datetime(verified_df['timestamp'])
verified_df['sighting_date'] = verified_df['parsed_time'].dt.date
verified_df['hour'] = verified_df['parsed_time'].dt.hour

# Group chronologically by calendar date
daily_departures = verified_df.sort_values('parsed_time').groupby('sighting_date').first()
daily_returns = verified_df.sort_values('parsed_time').groupby('sighting_date').last()

top_departure_district = daily_departures['district'].value_counts().index[0]
top_return_district = daily_returns['district'].value_counts().index[0]

print("[SPIDER-SENSE TELEMETRY] 🕵️ PHASE 1: CHRONOMETRIC DIRECTIONAL TELEMETRY")
print(f"  🌅 Primary Departure District (Start of Day): {format_title(top_departure_district)} ({daily_departures['district'].value_counts().iloc[0]} days)")
print(f"  🌃 Primary Return District (End of Day):     {format_title(top_return_district)} ({daily_returns['district'].value_counts().iloc[0]} days)")

# Plot First & Last Appearance Distributions
dep_counts = daily_departures['district'].value_counts().head(7)
ret_counts = daily_returns['district'].value_counts().head(7)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Morning Patrol Departures
dep_labels = [format_title(d) for d in dep_counts.index]
dep_colors = [SPIDEY_RED if 'Brooklyn_Central_0' in d else '#95A5A6' for d in dep_counts.index]
bars_dep = axes[0].barh(range(len(dep_counts)), dep_counts.values, color=dep_colors, edgecolor=SYMB_BLACK, linewidth=1.2)
axes[0].set_yticks(range(len(dep_counts)))
axes[0].set_yticklabels(dep_labels, fontsize=10, fontweight='bold')
axes[0].invert_yaxis()
axes[0].set_xlabel('Number Of Morning Departures (First Sighting)', fontsize=11, fontweight='bold')
axes[0].set_title('Morning Patrol Departures By District (Sunrise)', fontsize=12, fontweight='bold', pad=12)
for bar in bars_dep:
    w = bar.get_width()
    axes[0].text(w + 0.8, bar.get_y() + bar.get_height()/2, f'{int(w)} days', va='center', fontsize=9, fontweight='bold')

# Subplot 2: Nighttime Patrol Returns
ret_labels = [format_title(d) for d in ret_counts.index]
ret_colors = [WEB_BLUE if 'Brooklyn_Central_0' in d else '#95A5A6' for d in ret_counts.index]
bars_ret = axes[1].barh(range(len(ret_counts)), ret_counts.values, color=ret_colors, edgecolor=SYMB_BLACK, linewidth=1.2)
axes[1].set_yticks(range(len(ret_counts)))
axes[1].set_yticklabels(ret_labels, fontsize=10, fontweight='bold')
axes[1].invert_yaxis()
axes[1].set_xlabel('Number Of Nighttime Returns (Last Sighting)', fontsize=11, fontweight='bold')
axes[1].set_title('Nighttime Patrol Returns By District (Bedtime)', fontsize=12, fontweight='bold', pad=12)
for bar in bars_ret:
    w = bar.get_width()
    axes[1].text(w + 0.8, bar.get_y() + bar.get_height()/2, f'{int(w)} days', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

#### Phase 2: Nocturnal Rest Window Analysis (01:00 AM – 05:00 AM Sleep Hours)

Midday sightings are dominated by active criminal incidents in Manhattan. To isolate where Spider-Man rests, we inspect sightings between **01:00 AM and 05:00 AM** (deep sleep hours) and examine the 24-hour diurnal rhythm of `Brooklyn Central 0`.

In [ ]:
# Phase 2: Nocturnal Rest Hours Concentration (01:00 To 05:00 AM)
night_df = verified_df[(verified_df['hour'] >= 1) & (verified_df['hour'] <= 5)]
night_counts = night_df['district'].value_counts().head(7)

# 24-Hour Diurnal Sighting Distribution in Brooklyn Central 0
bk_hours = verified_df[verified_df['district'] == 'Brooklyn_Central_0']['hour'].value_counts().reindex(range(24), fill_value=0)

print("[SPIDER-SENSE TELEMETRY] 🕵️ PHASE 2: NOCTURNAL REST WINDOW TELEMETRY")
print(f"  🛌 Verified Sightings during 01:00–05:00 AM in {format_title(top_departure_district)}: {night_counts.iloc[0]} sightings")
print(f"  🥈 Runner-Up District ({format_title(night_counts.index[1])}): {night_counts.iloc[1]} sightings")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Resting Hours by District
night_labels = [format_title(d) for d in night_counts.index]
night_colors = [SPIDEY_RED if 'Brooklyn_Central_0' in d else '#95A5A6' for d in night_counts.index]
bars_night = axes[0].barh(range(len(night_counts)), night_counts.values, color=night_colors, edgecolor=SYMB_BLACK, linewidth=1.2)
axes[0].set_yticks(range(len(night_counts)))
axes[0].set_yticklabels(night_labels, fontsize=10, fontweight='bold')
axes[0].invert_yaxis()
axes[0].set_xlabel('Verified Sightings During Sleep Hours (01:00 To 05:00 AM)', fontsize=11, fontweight='bold')
axes[0].set_title('Nocturnal Rest Hours Concentration (01:00 To 05:00 AM)', fontsize=12, fontweight='bold', pad=12)
for bar in bars_night:
    w = bar.get_width()
    axes[0].text(w + 5, bar.get_y() + bar.get_height()/2, f'{int(w)} sightings', va='center', fontsize=9, fontweight='bold')

# Subplot 2: 24-Hour Diurnal Patrol Curve
axes[1].plot(range(24), bk_hours.values, 'o-', color=SPIDEY_RED, linewidth=2.5, markersize=6, markerfacecolor=SPIDER_GOLD, markeredgecolor=SYMB_BLACK)
axes[1].fill_between(range(24), bk_hours.values, alpha=0.25, color=SPIDEY_RED)
axes[1].axvspan(1, 5, color=WEB_BLUE, alpha=0.2, label='Nocturnal Rest Window (01:00 - 05:00 AM)')
axes[1].axvspan(22, 23.9, color='#8E44AD', alpha=0.15, label='Night Return Surge (10 PM - Midnight)')
axes[1].set_xticks(range(0, 24, 2))
axes[1].set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)], fontsize=9, fontweight='bold')
axes[1].set_xlabel('Hour Of Day (24-Hour Military Time)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Verified Sightings Count', fontsize=11, fontweight='bold')
axes[1].set_title('Brooklyn Central Diurnal Patrol Rhythm (24-Hour Cycle)', fontsize=12, fontweight='bold', pad=12)
axes[1].legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

#### Phase 3: Tactical Concealment Demographics (`locations.csv`)

Could `Brooklyn Central 0` be an industrial shipyard, a police precinct, or an open park? We cross-reference the environmental profile in `locations.csv` to evaluate:
- **Residential Score:** High concentration of apartments and houses to provide window and fire escape cover.
- **NYPD Patrol Density:** Low police scanner presence to avoid surveillance when taking off the mask.

In [ ]:
# Phase 3: Tactical Concealment Demographics from locations.csv
loc_csv_path = 'locations.csv'
if not os.path.exists(loc_csv_path):
    for root, dirs, files in os.walk('.'):
        if 'locations.csv' in files:
            loc_csv_path = os.path.join(root, 'locations.csv')
            break
if not os.path.exists(loc_csv_path) and 'path' in globals():
    for root, dirs, files in os.walk(path):
        if 'locations.csv' in files:
            loc_csv_path = os.path.join(root, 'locations.csv')
            break

loc_df = pd.read_csv(loc_csv_path)
bk_row = loc_df[loc_df['district'] == 'Brooklyn_Central_0'].iloc[0]

print("[SPIDER-SENSE TELEMETRY] 🕵️ PHASE 3: DISTRICT CONCEALMENT DOSSIER")
print(f"  🏙️ Target District:          {format_title(bk_row['district'])} ({bk_row['borough']})")
print(f"  🏠 Residential Score:        {bk_row['residential_score']:.3f} (Quiet housing and apartment density)")
print(f"  👮 NYPD Patrol Activity:     {bk_row['patrol_activity_score']:.3f} (Low police surveillance footprint)")

plt.figure(figsize=(11, 7))
plt.scatter(loc_df['patrol_activity_score'], loc_df['residential_score'],
            c='#BDC3C7', s=120, edgecolors=SYMB_BLACK, alpha=0.7, label='Other NYC Districts')

plt.scatter(bk_row['patrol_activity_score'], bk_row['residential_score'],
            c=SPIDEY_RED, s=380, marker='*', edgecolors=SYMB_BLACK, linewidth=2,
            label=f'Secret Sanctuary: Brooklyn Central 0\n(Residential: {bk_row["residential_score"]:.3f} | NYPD Patrol: {bk_row["patrol_activity_score"]:.3f})',
            zorder=10)

# Optimal quadrant boundary
plt.axhline(0.7, color=WEB_BLUE, linestyle='--', alpha=0.5)
plt.axvline(0.5, color=WEB_BLUE, linestyle='--', alpha=0.5)
plt.fill_between([0.0, 0.5], 0.7, 1.0, color='green', alpha=0.08, label='Optimal Vigilante Concealment Zone\n(High Residential Cover & Low NYPD Presence)')

plt.annotate('High Residential Cover + Low Police Scrutiny\n(Bedroom Window Access Without Detection)',
             xy=(bk_row['patrol_activity_score'], bk_row['residential_score']),
             xytext=(bk_row['patrol_activity_score'] - 0.18, bk_row['residential_score'] + 0.06),
             fontsize=9, fontweight='bold', color=SYMB_BLACK,
             arrowprops=dict(facecolor=SPIDEY_RED, arrowstyle='->', lw=1.5))

plt.xlabel('NYPD Patrol Activity Score (Surveillance & Scanner Risk)', fontsize=11, fontweight='bold')
plt.ylabel('Residential Housing Score (Apartment & Rooftop Cover)', fontsize=11, fontweight='bold')
plt.title('Tactical Concealment Demographics — Residential Cover Vs Police Surveillance', fontsize=13, fontweight='bold', pad=12)
plt.legend(fontsize=9.5, loc='lower left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

#### Phase 4: Precision Spatial Centroid Triangulation (Median $L_1$ Optimization)

The district of `Brooklyn Central 0` covers several square kilometers. To pinpoint the exact street coordinates, we extract all candidate departure points ($t_{\text{first}}$) and return points ($t_{\text{last}}$) belonging to this district.

Rather than taking the arithmetic mean (which can be skewed by chases toward district boundaries), we compute the **spatial component-wise median**:
$$\text{Lat}^* = \operatorname{median}(\text{latitude}), \quad \text{Lon}^* = \operatorname{median}(\text{longitude})$$
The median optimizes the $L_1$ norm (Manhattan distance) and possesses a 50% breakdown point, ensuring extreme geographic robustness.

In [ ]:
# Phase 4: Spatial Centroid Triangulation and Tactical Map
base_candidates = pd.concat([
    daily_departures[daily_departures['district'] == top_departure_district],
    daily_returns[daily_returns['district'] == top_return_district]
])

base_lat = base_candidates['latitude'].median()
base_lon = base_candidates['longitude'].median()

print("[SPIDER-SENSE TELEMETRY] 🎯 PINPOINT TRIANGULATED SANCTUARY:")
print(f"  District:   {format_title(top_departure_district)}")
print(f"  Borough:    {format_title(base_candidates['borough'].iloc[0])}")
print(f"  Latitude:   {base_lat:.5f}° N")
print(f"  Longitude:  {base_lon:.5f}° W")
print(f"  Candidates: {len(base_candidates)} boundary sightings analyzed")

# Two-Panel Tactical Map: Citywide Overview & Precision Zoom
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Panel 1: Citywide Overview
axes[0].scatter(coords['longitude'], coords['latitude'],
                c='#BDC3C7', alpha=0.12, s=3, label='Public Citizen Sightings (All NYC)')
axes[0].scatter(verified_df['longitude'], verified_df['latitude'],
                c=WEB_BLUE, alpha=0.25, s=5, label='Verified Spider-Man Sightings')
axes[0].scatter(base_lon, base_lat, c=SPIDER_GOLD, marker='*', s=500,
                edgecolors=SYMB_BLACK, linewidth=2.5,
                label=f'Secret Sanctuary: Brooklyn Central ({base_lat:.4f}°, {base_lon:.4f}°)',
                zorder=12)
axes[0].set_xlabel('Geographic Longitude', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Geographic Latitude', fontsize=11, fontweight='bold')
axes[0].set_title('Citywide Tactical Patrol Map And Sanctuary Triangulation', fontsize=12, fontweight='bold', pad=12)
axes[0].legend(fontsize=9.5, loc='upper right')

# Panel 2: Zoomed-in Brooklyn Central Cluster with Boundary Crosshairs
bk_dep = daily_departures[daily_departures['district'] == top_departure_district]
bk_ret = daily_returns[daily_returns['district'] == top_return_district]

axes[1].scatter(bk_dep['longitude'], bk_dep['latitude'],
                c=SPIDEY_RED, s=65, alpha=0.75, edgecolors=SYMB_BLACK,
                label=f'Morning Patrol Departures (N = {len(bk_dep)})')
axes[1].scatter(bk_ret['longitude'], bk_ret['latitude'],
                c=WEB_BLUE, s=65, alpha=0.75, edgecolors=SYMB_BLACK,
                label=f'Nighttime Patrol Returns (N = {len(bk_ret)})')

# Centroid crosshairs & gold star
axes[1].axvline(base_lon, color=SPIDER_GOLD, linestyle='--', linewidth=1.5, alpha=0.7)
axes[1].axhline(base_lat, color=SPIDER_GOLD, linestyle='--', linewidth=1.5, alpha=0.7)
axes[1].scatter(base_lon, base_lat, c=SPIDER_GOLD, marker='*', s=650,
                edgecolors=SYMB_BLACK, linewidth=2.5,
                label=f'Triangulated Sanctuary Centroid:\n{base_lat:.5f}° N, {base_lon:.5f}° W',
                zorder=15)

axes[1].set_xlabel('Geographic Longitude (Zoomed: Brooklyn Sector)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Geographic Latitude (Zoomed: Brooklyn Sector)', fontsize=11, fontweight='bold')
axes[1].set_title('Precision Spatial Centroid Triangulation (Median L1 Optimization)', fontsize=12, fontweight='bold', pad=12)
axes[1].legend(fontsize=9.5, loc='upper right')

plt.tight_layout()
plt.show()

#### Mission 05 Solved: Final Detective Dossier

Through multi-factor chronometric behavioral triangulation, the unscored mystery is decisively solved:

| Telemetry Factor | Empirical Metric | Behavioral / Analytical Significance |
|---|---|---|
| **Designated Base District** | `Brooklyn Central 0` | Borough of Brooklyn (Bensonhurst / Bay Ridge area) |
| **Pinpoint Coordinates** | **`40.5898° N, -74.0170° W`** | Triangulated spatial $L_1$ median centroid |
| **Morning Departures ($t_0$)** | **45 Days** | #1 origin across all NYC districts (sunrise patrol starts) |
| **Nighttime Returns ($t_{\text{end}}$)** | **35 Nights** | #1 terminal across all NYC districts (bedtime patrol concludes) |
| **Sleep Hours (01:00–05:00 AM)** | **445 Sightings** | Stationary rooftop clustering during deep nocturnal hours |
| **Residential Cover Score** | **`0.838`** | High residential apartment / home density for stealth entry |
| **NYPD Patrol Score** | **`0.412`** | Low police radar surveillance prevents unmasking |

**Final Deduction:** Spider-Man operates his secret sanctuary from a residential apartment bedroom in **southern Brooklyn** (`Brooklyn Central 0`, `40.5898° N, -74.0170° W`), swinging out through a quiet alley or window before patrolling north into Manhattan.

---

## 7. Progressive Suit Upgrade Log (Model Evolution)

Here is the complete record of every optimization cycle, demonstrating quantifiable improvement from the baseline prototype to the final Stark-Tech model.

In [ ]:
results_df = pd.DataFrame(results)
results_df.index = range(1, len(results_df) + 1)
results_df['Gain'] = results_df['Accuracy'].diff().fillna(0)
results_df['Gain_Str'] = results_df['Gain'].apply(lambda x: f"+{x:.4f}" if x >= 0 else f"{x:.4f}")

print("=" * 90)
print("🕷️ SPIDEY-TRACKER: SUIT UPGRADE & OPTIMIZATION CHRONOLOGY")
print("=" * 90)
for idx, row in results_df.iterrows():
    print(f"  [{row['Step']:42s}] | Accuracy: {row['Accuracy']:.4f} ({row['Gain_Str']}) | {row['Details']}")
print("=" * 90)
total_gain = results_df['Accuracy'].iloc[-1] - results_df['Accuracy'].iloc[0]
print(f"🚀 Total Performance Evolution: {results_df['Accuracy'].iloc[0]:.4f} → {results_df['Accuracy'].iloc[-1]:.4f} ({'+' if total_gain >= 0 else ''}{total_gain:.4f} net boost)")

In [ ]:
# Visualizing Progressive Accuracy Gains
fig, ax = plt.subplots(figsize=(14, 6))

suit_colors = [
    '#95A5A6',  # Homemade Hoodie (Gray)
    '#3498DB',  # Classic (Blue)
    '#2980B9',  # Advanced Shooters
    '#8E44AD',  # MK I Armor
    '#16A085',  # Stealth Big Time
    '#E67E22',  # Superior Spider
    SPIDEY_RED  # Iron Spider Stark Suit
]

bars = ax.bar(range(len(results_df)), results_df['Accuracy'], color=suit_colors,
              edgecolor=SYMB_BLACK, linewidth=1.2, width=0.6)

for i, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f"{results_df['Accuracy'].iloc[i]:.4f}\n({results_df['Gain_Str'].iloc[i]})",
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(results_df)))
ax.set_xticklabels(results_df['Step'], rotation=30, ha='right', fontsize=9, fontweight='bold')
ax.set_xlabel('Suit Upgrade Stage', fontsize=12, fontweight='bold')
ax.set_ylabel('Spider-Sense Model Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Spider-Sense Calibration Trajectory — Progressive Suit Upgrades', fontsize=14, fontweight='bold', pad=12)

y_min = max(0, results_df['Accuracy'].min() - 0.06)
y_max = min(1, results_df['Accuracy'].max() + 0.04)
ax.set_ylim(y_min, y_max)
plt.tight_layout()
plt.show()

In [ ]:
# Trajectory Line Chart
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(range(len(results_df)), results_df['Accuracy'], 'o-', color=SYMB_BLACK,
        markersize=11, linewidth=2.5, markerfacecolor=SPIDEY_RED, markeredgecolor='white', markeredgewidth=2)

for i, row in results_df.iterrows():
    ax.annotate(f"{row['Accuracy']:.4f}",
                (i, row['Accuracy']),
                textcoords="offset points", xytext=(0, 16),
                ha='center', fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=SYMB_BLACK, lw=0.8))

ax.set_xticks(range(len(results_df)))
ax.set_xticklabels(results_df['Step'], rotation=30, ha='right', fontsize=9, fontweight='bold')
ax.set_xlabel('Suit Upgrade Cycle', fontsize=11, fontweight='bold')
ax.set_ylabel('Model Accuracy Score', fontsize=11, fontweight='bold')
ax.set_title('Spider-Sense Learning Curve Across 7 Optimization Cycles', fontsize=13, fontweight='bold', pad=12)
ax.set_ylim(y_min, y_max)
plt.tight_layout()
plt.show()

## 8. Final Dossier: Friendly Neighborhood Machine Learning

**1. K-NN Spider-Sense Classifier:**
- Progressed systematically across 7 suit-upgrade cycles from a raw baseline to Stark-Tech grade performance.
- Key catalysts of accuracy gain:
  - **StandardScaler Normalization:** Neutralized multi-thousand-meter scale distortions.
  - **K-Tuning & Manhattan Topology:** Aligned algorithm geometry with New York City's street grid.
  - **ANOVA Sensory Pruning:** Filtered Mysterio noise and isolated top predictive signals.

**2. K-Means Patrol Grid:**
- Discovered natural high-frequency patrol sectors across Manhattan, Brooklyn, and Queens using Elbow and Silhouette analysis.

**3. Mission 05 Solved:**
- Triangulated Spider-Man's secret sanctuary in **Brooklyn Central** (`~40.5898° N, -74.0170° W`), supported by daily departure and return telemetry and quiet neighborhood demographics.